# Is a Network Worth It for Pricing Homes? — Assignment

**Week 6 · Assignment 6** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 6*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week6/assignments/week06_pricing_network_tabpfn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lessons 7 and 9 — the network you trained on churn, and the pretrained
TabPFN that tied Week 5's XGBoost — on a different problem: Week 3's home-pricing data.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_housing
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range, check_shape, check_type

## Before You Start

- **Format — same shape every week:** **A. Extend** (40 pts, scaffolded
  fill-ins on the week's mechanic) → **B. Decide** (30 pts, one modeling
  decision with a number and a justification) → **C. Recommend** (30 pts, a
  stakeholder-facing writeup). Edit only the `# YOUR CODE HERE` blocks and
  the *\<placeholder\>* prose — everything else is given.
- **Part A is mostly yours this week.** The data, split, and earlier models are given;
  you write the network, its training, and the TabPFN fit, using Lessons 6, 7, and 9.
- **Self-checks are feedback, not the grade.** The `assert` cells after each Part A
  fill-in are checkpoints: green output means you're on track.
- **Runtime:** plan on 10–15 minutes for *Restart & Run All* on Colab. Most of that is
  TabPFN, which reads its training rows for every prediction (Lesson 8) and can take
  5–10 minutes on its own; the six small networks take a few minutes together. A
  long-running cell isn't stuck.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb` and submit
  that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus). Parts B and C are graded on judgment against
  *your own* results, so ground your answers in the numbers your own run produced.

## Summary of Answers

*Fill this in last, after finishing Parts B and C below.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** | _Your answer here._ |
| **C — Recommend** | _Your answer here._ |

## 1. Business Problem

Week 3's assignment asked how simple the appraisal team's pricing model could get, and
the cross-validated lasso priced test homes with an RMSE of about \$28,000. This week's
lessons built two kinds of neural network: one trained from scratch (Lesson 7) and one
pretrained on synthetic tables (TabPFN, Lesson 9) — and on churn, neither beat Week 5's
tuned XGBoost. This assignment asks the same question on a different problem.

**Business question:** should the appraisal team price homes with a neural network, a
pretrained model like TabPFN, or a tuned tree ensemble — and is any gain worth what the
fancier model costs?

**Why it matters:** every dollar of pricing error is a home listed too high (it sits) or
too low (the seller loses money); a model that cuts the typical error is worth real
money — but only if it can be run, explained, and trusted.

## 2. Data — Part A: Extend

*Same housing data, columns, and split as Week 3's assignment, so every number lines up
with Week 3's. `load_housing()` fills every blank, so there are no missing values.*

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

df = load_housing()
X, y = df.drop(columns="SalePrice"), df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# Week 3's encoder: one 0/1 column per category, rare ones pooled
encode = make_column_transformer(
    (OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist", sparse_output=False),
     make_column_selector(dtype_exclude="number")),
    remainder="passthrough", verbose_feature_names_out=False,
)

# A validation slice from the training rows only, for early stopping and Part B
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_SEED)
print(f"{len(df):,} homes · fit {len(X_fit):,} / validation {len(X_val):,} / test {len(X_test):,}")

**Prepare the network's inputs.** A network trains by gradient descent (Lesson 5), so
both sides need scaling (Lesson 7's notebook scaled the features; a price target
needs it too):

- **Features:** encode, then standardize. Fit `prep` on **`X_fit` only** — define in
  Data, fit only on training rows — and transform all three slices into float arrays
  `Xf`, `Xv`, `Xt`.
- **Target:** sale prices run to six figures, and a network trained on raw dollars
  learns badly. Standardize it with the **fit rows'** mean and standard deviation
  (`y_mean`, `y_sd`): `yf = (y_fit - y_mean) / y_sd`, and the same for `yv`. To turn a
  prediction back into dollars, reverse it: `pred * y_sd + y_mean`.

In [ ]:
# YOUR CODE HERE

print(f"network inputs: {Xf.shape[1]} columns")

In [ ]:
check_shape(Xf, (len(X_fit), Xf.shape[1]), "Xf has one row per fit home")
check_equal(Xv.shape[1] == Xf.shape[1] == Xt.shape[1], True, "all three slices have the same columns")
check_close(float(np.abs(Xf.mean(axis=0)).max()), 0.0, "features are standardized on the fit rows", atol=1e-3)
check_close(float(yf.mean()), 0.0, "target is centered using the fit rows", atol=1e-6)
check_close(float(yf.std()), 1.0, "target is scaled to unit spread using the fit rows", atol=0.01)

## 3. Model — Part A: Extend

**What to beat.** The table below is given: Week 3's models, and XGBoost run here with
Week 5's recipe. XGBoost's settings come from Week 5's 100-candidate cross-validated
search on these training rows; the search itself takes over 10 minutes, so it isn't
repeated — only the final model is refit.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
ols = make_pipeline(encode, StandardScaler(), LinearRegression()).fit(X_train, y_train)
lasso = make_pipeline(encode, StandardScaler(), Lasso(alpha=1000, max_iter=10_000)).fit(X_train, y_train)  # Week 3's CV-chosen λ
xgb_default = make_pipeline(encode, XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.1,
                                                 random_state=RANDOM_SEED)).fit(X_train, y_train)
xgb_tuned = make_pipeline(encode, XGBRegressor(n_estimators=3128, learning_rate=0.01, max_depth=5, min_child_weight=1, subsample=0.6, colsample_bynode=0.6, gamma=1, reg_lambda=5, reg_alpha=1, random_state=RANDOM_SEED)).fit(X_train, y_train)

test_pred = {
    "Baseline (average price)": baseline.predict(X_test),
    "OLS, all fields (Week 3)": ols.predict(X_test),
    "Lasso, CV-chosen (Week 3)": lasso.predict(X_test),
    "XGBoost, defaults (Week 5 recipe)": xgb_default.predict(X_test),
    "XGBoost, tuned (Week 5 recipe)": xgb_tuned.predict(X_test),
}
for name, pred in test_pred.items():
    print(f"{name:36s} test RMSE ${root_mean_squared_error(y_test, pred):,.0f}")

**Your network.** Write `fit_network(l2)`: Lesson 7's network, adapted to a price.

- Two hidden layers, 64 and 32 ReLU units, each followed by 20% dropout, and weight
  decay `l2` on the hidden layers (`kernel_regularizer=keras.regularizers.l2(l2)`, or
  none when `l2` is 0).
- **One output unit with no activation** — a price, not a probability — and
  **mean squared error** as the loss (Lesson 4's pairing for a numeric target).
- Adam, batch size 32, up to 300 epochs, with early stopping on validation loss
  (patience 20, restoring the best weights). Call `keras.utils.set_random_seed(RANDOM_SEED)`
  first so runs repeat.
- Train on `Xf`/`yf`, validate on `Xv`/`yv`, and return the trained model.

Then fit it with no weight decay (`l2=0`) as `network_plain`, and add its test
predictions, **converted back to dollars**, to `test_pred` as `"Network, no weight decay"`.

In [ ]:
%pip install -q keras
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # quiet TensorFlow's startup info/warning/error logs

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")   # keep the networks on the CPU; restart the kernel to re-run

import keras
from keras import layers

# YOUR CODE HERE

print(f"Network, no weight decay: test RMSE ${root_mean_squared_error(y_test, test_pred['Network, no weight decay']):,.0f}")

In [ ]:
check_equal(network_plain.output_shape, (None, 1), "the network has one output: a price")
check_equal(network_plain.layers[-1].activation.__name__, "linear", "the output has no activation (not a probability)")
check_equal(len(test_pred["Network, no weight decay"]), len(X_test), "one predicted price per test home")
check_in_range(float(np.median(test_pred["Network, no weight decay"])), 100_000, 300_000,
               "predictions were converted back to dollars")

**TabPFN.** Fit Lesson 9's pretrained model as a **regressor** this time:
`TabPFNRegressor.create_default_for_version(ModelVersion.V2, n_estimators=2, device="auto",
random_state=RANDOM_SEED)`. TabPFN takes the raw columns (no encoding or scaling) — just
convert the text columns to pandas `category` first, as Lesson 9 did. Fit on
`X_train`/`y_train`, predict the test homes, and add them to `test_pred` as
`"TabPFN v2 (no training, no tuning)"`. On Colab, expect this cell to take 5–10
minutes.

In [ ]:
%pip install -q tabpfn==9.2.0
import os
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# YOUR CODE HERE


In [ ]:
check_equal(len(test_pred["TabPFN v2 (no training, no tuning)"]), len(X_test), "one TabPFN price per test home")
check_equal(root_mean_squared_error(y_test, test_pred["TabPFN v2 (no training, no tuning)"])
            < root_mean_squared_error(y_test, test_pred["Baseline (average price)"]), True, "TabPFN beats the baseline")

## 4. Evaluate — Part A: Extend

*Every model, scored on the same test homes — given, using your `test_pred` entries.*

In [ ]:
scores = pd.Series({name: root_mean_squared_error(y_test, pred) for name, pred in test_pred.items()},
                   name="test RMSE ($)").sort_values()
lasso_rmse = scores["Lasso, CV-chosen (Week 3)"]
pd.DataFrame({"test RMSE ($)": scores.round(0),
              "vs. lasso": (100 * (scores / lasso_rmse - 1)).round(1).astype(str) + "%"})

**Is the top of the table a real ranking?** TabPFN and tuned XGBoost were scored on
the same homes, so compare them *paired* (Lesson 9's bootstrap, applied to the difference
between two models): resample test homes
1,000 times and look at the middle 95% of the average difference in squared error. If it
straddles zero, this test set can't tell them apart.

In [ ]:
def paired_bootstrap(a, b, n_boot=1000):
    """95% interval for the RMSE difference (a − b), resampling test homes."""
    ea, eb = (test_pred[a] - y_test.to_numpy()) ** 2, (test_pred[b] - y_test.to_numpy()) ** 2
    draws = rng.integers(0, len(ea), size=(n_boot, len(ea)))
    diffs = np.sqrt(ea[draws].mean(axis=1)) - np.sqrt(eb[draws].mean(axis=1))
    return np.percentile(diffs, [2.5, 97.5])


for a, b in [("TabPFN v2 (no training, no tuning)", "XGBoost, tuned (Week 5 recipe)"),
             ("Network, no weight decay", "Lasso, CV-chosen (Week 3)")]:
    low, high = paired_bootstrap(a, b)
    print(f"{a} − {b}: ${scores[a] - scores[b]:+,.0f}  (95% interval ${low:+,.0f} to ${high:+,.0f})")

## 5. Interpret — Part B: Decide

Part A's network used no weight decay. Choose it properly: try **`l2` in
{0, 0.0001, 0.001, 0.01, 0.1}**, score each network's **validation** RMSE in dollars (the
test set stays out of the choice), and pick the value with the lowest. Store the validation RMSEs as a pandas Series
`val_rmse` (indexed by `l2`), the winner as `chosen_l2`, and its test RMSE as
`chosen_test_rmse`; add its test predictions to `test_pred` as `"Network, your weight decay"`.

In [ ]:
# YOUR CODE HERE

print(val_rmse.round(0).to_string())
print(f"\nChosen weight decay: {chosen_l2:g}  ->  test RMSE ${chosen_test_rmse:,.0f} "
      f"(lasso ${lasso_rmse:,.0f})")

**Your answer:** *\<the weight decay you chose, and its validation and test RMSE\>*

**Justification (3–5 sentences):** _Your answer here._ 

## 6. Recommend — Part C: Recommend

*Stakeholder-facing. Write for the appraisal team lead deciding which model prices
homes — reference your own numbers, including the given rows.*

**Recommendation:** _Your answer here._

**What would change this recommendation:** _Your answer here._

**Limits:** _Your answer here._

## Rubric

*Same four criteria, same point split, every week — 100 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's weight decay is the one your own validation RMSE supports. |
| **Justification** | 20 | Part B's reasoning (10 pts) engages with your own numbers; Part C's recommendation (10 pts) follows logically from your own Part B answer. |
| **Communication** | 20 | Part C reads as a stakeholder memo — clear, actionable, and honest about limits. |